<a href="https://colab.research.google.com/github/mmferes/PPD-2026/blob/main/Aula3/praticaguiada_2403_1_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Identificação de Hotspots em Programas Sequenciais
Antes de paralelizar um programa, é fundamental responder a seguinte pergunta:

**Quais partes do código realmente consomem tempo?**

Nem todo trecho de um programa contribui igualmente para o tempo total de execução. Em muitos casos, uma pequena parte do código (hotspot) é responsável pela maior parte do custo computacional.

De acordo com boas práticas de projeto de programas paralelos, o esforço de paralelização deve ser concentrado nesses trechos críticos, pois são eles que oferecem maior potencial de ganho de desempenho.

Nesta atividade, utilizaremos uma ferramenta de profiling para identificar esses pontos críticos em um programa em C.

## [gprof](https://ftp.gnu.org/old-gnu/Manuals/gprof-2.9.1/html_mono/gprof.html#SEC2): ferramenta para análise de desempenho em programas
Ao executar um programa, nem todas as funções contribuem igualmente para o tempo total. Em muitos casos, uma pequena fração do código — chamada de hotspot — é responsável pela maior parte do custo computacional.

O gprof permite:

* Identificar funções mais custosas;
* Medir tempo de execução por função;
* Contar quantas vezes cada função foi chamada;
* Analisar a relação entre funções (quem chama quem);

Essas informações são fundamentais para decidir onde concentrar esforços de otimização ou paralelização.

Apesar de útil, o **gprof possui algumas limitações**:

* Não mede bem programas com múltiplas threads;
* Pode introduzir overhead na execução;
* Não fornece informações detalhadas de hardware (cache, memória, etc.)

Para análises mais avançadas, outras ferramentas podem ser utilizadas, como perf, Valgrind ou ferramentas específicas de HPC.

## O problema
O programa abaixo realiza duas operações principais:

1.  Multiplicação de matrizes; e
2.  Soma dos elementos da matriz resultante

A multiplicação de matrizes é um problema clássico em computação científica e frequentemente utilizado em estudos de paralelização.

In [ ]:
%%writefile hotspot.c

#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define N 800

double A[N][N], B[N][N], C[N][N];

void gerar_matrizes() {
    for(int i=0;i<N;i++)
        for(int j=0;j<N;j++) {
            A[i][j] = rand()%100;
            B[i][j] = rand()%100;
        }
}

void multiplicacao() {
    for(int i=0;i<N;i++)
        for(int j=0;j<N;j++)
            for(int k=0;k<N;k++)
                C[i][j] += A[i][k]*B[k][j];
}

double soma() {
    double s = 0;
    for(int i=0;i<N;i++)
        for(int j=0;j<N;j++)
            s += C[i][j];
    return s;
}

int main() {
    gerar_matrizes();

    clock_t inicio = clock();

    multiplicacao();
    double s = soma();

    clock_t fim = clock();

    printf("Soma: %f\n", s);
    printf("Tempo: %f\n", (double)(fim - inicio)/CLOCKS_PER_SEC);

    return 0;
}

Overwriting hotspot.c


In [ ]:
! if [ ! -f hotspot ]; then gcc -pg hotspot.c -o hotspot; fi;
!./hotspot


Soma: 1255382499470.000000
Tempo: 7.573748


In [ ]:
!gprof hotspot gmon.out

Flat profile:

Each sample counts as 0.01 seconds.
  %   cumulative   self              self     total           
 time   seconds   seconds    calls   s/call   s/call  name    
 99.87      7.61     7.61        1     7.61     7.61  multiplicacao
  0.13      7.62     0.01        1     0.01     0.01  gerar_matrizes
  0.00      7.62     0.00        1     0.00     0.00  soma

 %         the percentage of the total running time of the
time       program used by this function.

cumulative a running sum of the number of seconds accounted
 seconds   for by this function and those listed above it.

 self      the number of seconds accounted for by this
seconds    function alone.  This is the major sort for this
           listing.

calls      the number of times this function was invoked, if
           this function is profiled, else blank.

 self      the average number of milliseconds spent in this
ms/call    function per call, if this function is profiled,
	   else blank.

 total     the aver

#Responda
1. Este problema pode ser paralelizado?
2. Qual estratégia inicia? Decomposição de Domínio ou de Funcional?
3. Qual granularidade? Fina ou grossa?

# Conclusão
1. A análise de desempenho é etapa essencial antes da paralelização.
2. Os Hotspots determinam o ganho possível;
3. Nem todo código deve ser paralelizado;
4. Problemas com alta intensidade computacional e baixa dependência são ideais.




# Atividade - faça você mesmo
Analise o programa abaixo e responda:
1. Qual função consome mais tempo?
2. Quantas vezes ela é chamada?
3. Por que ela domina o tempo de execução?
4. O problema pode ser paralelizado?
5. Como particionar?
6. Há dependência de dados?
7. Como ficaria o balanceamento?



```
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <time.h>

#define N 200000

// Função custosa (hotspot)
int eh_primo(int n) {
    if (n < 2) return 0;
    for (int i = 2; i <= sqrt(n); i++) {
        if (n % i == 0)
            return 0;
    }
    return 1;
}

// Função principal de cálculo
int conta_primos(int limite) {
    int count = 0;
    for (int i = 2; i < limite; i++) {
        if (eh_primo(i))
            count++;
    }
    return count;
}

int main() {
    clock_t inicio = clock();

    int total = conta_primos(N);

    clock_t fim = clock();

    printf("Total de primos: %d\n", total);
    printf("Tempo: %f segundos\n",
           (double)(fim - inicio)/CLOCKS_PER_SEC);

    return 0;
}```

